#	Importing the libraries


In [408]:
import requests
import re
import numpy as np
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.corpus import wordnet as wn
from collections import defaultdict
from bs4 import BeautifulSoup

#	Building The Index

In [410]:
class buildIndexClass:
    docs = [] 
    wordIndex={}
    preprocessed_docs = {} 
    stop_words = set(stopwords.words('english'))
    stemmer = PorterStemmer()

    #get the stem of the word
    def get_stem(self,word):
        return self.stemmer.stem(word)

    #performing preprocessing, especially: converting to lowercase, normalization, tokenization and stop words removal
    #for both the search query and the the index words while building it 
    
    #the actual process of building the positional inverted index
    def buildIndex (self,docs):
        termsIndex={}
        for doc_id, doc in enumerate(docs):
            docsSplit = doc.split()
            preprocessed_terms = []
            for position, word in enumerate(docsSplit):
               word=word.lower()
               word=self.get_stem(word)
               if word not in self.stop_words:
                preprocessed_terms.append(word)
                
                if word not in termsIndex:
                    termsIndex[word] = {}
               
                if doc_id not in termsIndex[word]:
                    termsIndex[word][doc_id] = []
                termsIndex[word][doc_id].append(position)    
            self.preprocessed_docs[doc_id] = preprocessed_terms
        return termsIndex  
    def buildFinalIndex(self):
        self.wordIndex=self.buildIndex(self.docs)
        # rank_instance = rankClass()
        # rank_instance.build_tfidf()

# Crawling The Website

In [412]:
class crawlerClass:

    def __init__(self):
        self.to_visit = list()
        self.visited = set()

    #fetching the content of the webpage
    def fetch(self, url):
        print('now fetching.. ' , url)
        res = requests.get(url).content
        return res
        
    #getting the next url to fetch
    def get_current_url(self):
        res = self.to_visit.pop()
        while res in self.visited:
            print('already visited',res)
            res = self.to_visit.pop()
        return res
        
    #getting all the links in the webpage
    def get_links(self , content , input_pattren ):
        urls = re.findall('<a href="([^"]+)">', str(content))
        print('urls are', urls)
        for url in urls:
            pattern = re.compile(input_pattren)
            if pattern.match(url):
                self.to_visit.append(url)

    #doing the actual crawling process from the root url
    def  crawl(self, url,input_pattren, depth=15):
        i=buildIndexClass()
        
        self.to_visit.append(url)
        while len(self.visited) <= depth:
            current_url = self.get_current_url()           
            content = self.fetch(current_url)
            soup = BeautifulSoup(content, "lxml")
            text_content=soup.get_text(strip=True)
            text_content = text_content.strip('\n')
            text_content.replace("\n","")
            print(text_content)
            i.docs.append(text_content)
            self.visited.add(current_url)
            self.get_links(content,input_pattren)
        print(self.to_visit)
        print(self.visited)

In [413]:
#Crawling Wikipedia
c = crawlerClass()
c.crawl('https://en.wikipedia.org/wiki','https?')

now fetching..  https://en.wikipedia.org/wiki
Wikipedia, the free encyclopediaJump to contentMain menuMain menumove to sidebarhideNavigationMain pageContentsCurrent eventsRandom articleAbout WikipediaContact usContributeHelpLearn to editCommunity portalRecent changesUpload fileSearchSearchAppearanceDonateCreate accountLog inPersonal toolsDonateCreate accountLog inPages for logged out editorslearn moreContributionsTalkMain PageMain PageTalkEnglishReadView sourceView historyToolsToolsmove to sidebarhideActionsReadView sourceView historyGeneralWhat links hereRelated changesUpload fileSpecial pagesPermanent linkPage informationCite this pageGet shortened URLDownload QR codePrint/exportDownload as PDFPrintable versionIn other projectsWikimedia CommonsWikimedia FoundationMediaWikiMeta-WikiWikimedia OutreachMultilingual WikisourceWikispeciesWikibooksWikidataWikifunctionsWikimaniaWikinewsWikiquoteWikisourceWikiversityWikivoyageWiktionaryWikidata itemAppearancemove to sidebarhideFrom Wikipedia,

# Implementing The Search Engine

In [415]:
class searchClass:
    #performing preprocessing, especially: converting to lowercase, normalization, tokenization and stop words removal
    #for both the search query and the the index words while building it 
    
    def __init__(self, index):
        self.index = index

    rank_instance = rankClass()
    rank_instance.build_tfidf()
    
    ind=buildIndexClass()

    #finding similiar words for a word in the wordnet
    def findInWordNet(self,word):
        similarWords=[]
        for w in wn.synsets(word):
            if w.lemma_names()[0] not in similarWords:
                similarWords.append(w.lemma_names()[0])
        return similarWords

    #performing a search operation on a word and doing preprocessing on it to match the preprocessing done while building the index
    #then return the results sorted by their TF-IDF scores and containing the positions for the terms
    def search(self, query):
        searchQuery = self.ind.get_stem(query)
        if searchQuery in self.index.keys():
            if searchQuery not in self.ind.stop_words:
                results = []
                for doc_id, positions in self.index[searchQuery].items():
                    tfidf_score = self.rank_instance.tfidf_scores[searchQuery].get(doc_id, 0)
                    results.append((doc_id, positions, tfidf_score))
                return sorted(results, key=lambda x: x[2], reverse=True)
        else:
            return []


    #or search operation between two words 
    def orSearch(self, t1, t2):
        res1 = self.search(t1)
        res2 = self.search(t2)
        #OR Operation between the two results res1 and res2
        resOr = list({(doc_id, tuple(positions), tfidf_score) for doc_id, positions, tfidf_score in (res1 + res2)}) 
        return sorted(resOr, key=lambda x: x[2], reverse=True)

    #and search operation between two words     
    def andSearch(self, t1, t2):
        res1 = self.search(t1)
        res2 = self.search(t2)
        #AND OPERARTION between the two results res1 and res2
        resAnd = [
            (doc_id, sorted(set(positions1 + positions2)), tfidf_score1 + tfidf_score2)
            for doc_id, positions1, tfidf_score1 in res1
            for doc_id2, positions2, tfidf_score2 in res2
            if doc_id == doc_id2
        ]
        return sorted(resAnd, key=lambda x: x[2], reverse=True)


    #not search operation between two words      
    def notSearch(self, t1, t2):
        res1 = self.search(t1)
        res2 = self.search(t2)
    
        #NOT OPERARTION
        resNot = [
            (doc_id, positions, tfidf_score)
            for doc_id, positions, tfidf_score in res1
            if doc_id not in {doc_id for doc_id, _, _ in res2}
        ]

        return sorted(resNot, key=lambda x: x[2], reverse=True)

        
    #performing a search operation on all words of a sentence        
    def searchSentence(self,sentence):
        #the function searches by doing "and" on all words of the sentence
        #if no results it does "or" operation on all words of the sentence
        #if still no results it find the simillar words in the WordNet then
        #it does an "and" operation and if still no results it does "or" operation
        #and then finally it shows the results (if they are non empty) ordered by their TF-IDF scores
        tempRes=list()
        res=list()
        finalRes=list()
        sentenceSplit=list()
        sentenceSplitWithStopWords=sentence.split()
        for i in sentenceSplitWithStopWords:
            if i not in self.ind.stop_words:
                sentenceSplit.append(i)
        for i in range(len(sentenceSplit)):
            sentenceSplit[i]=sentenceSplit[i].lower()
            sentenceSplit[i]=self.ind.get_stem(sentenceSplit[i])
        if len(sentenceSplit)==1:
            return self.search(sentenceSplit[0])
        for i in range(len(sentenceSplit)):
            if i-1>=(len(sentenceSplit)-len(sentenceSplit)) and i<len(sentenceSplit):
                resx=self.andSearch(sentenceSplit[i-1],sentenceSplit[i])
                if len(res) == 0:
                    res=resx
                else:
                  res = [item for item in res if item in resx]   
        if len(res) == 0:
            for i in range(len(sentenceSplit)):
                if i-1>=(len(sentenceSplit)-len(sentenceSplit)) and i<len(sentenceSplit):
                    resx=self.orSearch(sentenceSplit[i-1],sentenceSplit[i])
                    res.extend(resx)           
        if len(res) == 0:
         for i in range(len(sentenceSplit)):
             if i - 1 >= (len(sentenceSplit) - len(sentenceSplit)) and i< len(sentenceSplit):
                 word1Synonyms = self.findInWordNet(sentenceSplit[i - 1])
                 word2Synonyms = self.findInWordNet(sentenceSplit[i])
                 for word1 in word1Synonyms: 
                     for word2 in word2Synonyms:
                         resx = self.orSearch(word1, word2)
                         res.extend(resx) 
        if len(res) == 0:
         for i in range(len(sentenceSplit)):
             if i - 1 >= (len(sentenceSplit) - len(sentenceSplit)) and i< len(sentenceSplit):
                 word1Synonyms = self.findInWordNet(sentenceSplit[i - 1])
                 word2Synonyms = self.findInWordNet(sentenceSplit[i])
                 for word1 in word1Synonyms: 
                     for word2 in word2Synonyms:
                         resx = self.andSearch(word1, word2)
                         res.extend(resx)
        tfidf_scores_dic = {}
        for word in sentenceSplit:
                if word in rank_instance.tfidf_scores:
                    for doc_id, score in rank_instance.tfidf_scores[word].items():
                        if doc_id not in tfidf_scores_dic:
                            tfidf_scores_dic[doc_id] = 0
                        tfidf_scores_dic[doc_id] += score

        # order results by TF-IDF scores
        sorted_results = sorted(tfidf_scores_dic.items(), key=lambda x: x[1], reverse=False)
        finalRes = [doc_id for doc_id, _ in sorted_results]
    

        finalRes=set(res)  
        finalRes=list(finalRes)
        return finalRes

# Using TF-IDF Scores To Rank Search Results

In [417]:
class rankClass:
    def __init__(self):
        self.documents = []  # Preprocessed documents from buildIndexClass
        self.tfidf_scores = defaultdict(dict)  # {term: {doc_id: tf-idf score}}
    buildIndexInstance = buildIndexClass()

    #function to compute tf (Term Frequency)
    def compute_tf(self, word, doc_terms):
        freq = doc_terms.count(word)
        return freq / len(doc_terms) if len(doc_terms) > 0 else 0
    #function to compute idf (Inverse Document Frequency)
    def compute_idf(self, word, docs_terms):
        doc_count = sum(1 for terms in docs_terms.values() if word in terms)
        return np.log(len(docs_terms) / (1 + doc_count))
    #building and computing tf-idf scores for all words in the index
    def build_tfidf(self):
        docs_terms = self.buildIndexInstance.preprocessed_docs  # Use stored preprocessed terms
        self.documents = docs_terms

        for doc_id, terms in docs_terms.items():
            for term in terms:
                tf = self.compute_tf(term, terms)
                idf = self.compute_idf(term, docs_terms)
                self.tfidf_scores[term][doc_id] = tf * idf

In [421]:
build_index_instance = buildIndexClass()
build_index_instance.buildFinalIndex()
print(build_index_instance.wordIndex)
#printing the built positional inverted index

{'wikipedia,': {0: [0, 63], 3: [128], 7: [850, 1534, 2072]}, 'free': {0: [2, 65], 7: [387, 729, 924, 1028, 7537], 11: [372], 14: [370], 15: [561, 627]}, 'encyclopediajump': {0: [3]}, 'contentmain': {0: [5], 1: [9], 4: [5], 6: [6], 7: [10], 8: [12], 11: [9], 12: [4], 14: [9]}, 'menumain': {0: [6], 1: [10], 4: [6], 6: [7], 7: [11], 8: [13], 11: [10], 12: [5], 14: [10]}, 'menumov': {0: [7], 1: [11], 4: [7], 6: [8], 7: [12], 8: [14], 11: [11], 12: [6], 14: [11]}, 'sidebarhidenavigationmain': {0: [9], 4: [9], 6: [10], 12: [8]}, 'pagecontentscurr': {0: [10]}, 'eventsrandom': {0: [11]}, 'articleabout': {0: [12]}, 'wikipediacontact': {0: [13]}, 'uscontributehelplearn': {0: [14]}, 'editcommun': {0: [16]}, 'portalrec': {0: [17], 4: [19], 6: [20]}, 'changesupload': {0: [18, 40], 4: [58], 6: [60]}, 'filesearchsearchappearancedonatecr': {0: [19]}, 'accountlog': {0: [20, 23], 4: [27, 30], 6: [28, 31], 12: [22, 25]}, 'inperson': {0: [21], 1: [40], 4: [28], 6: [29], 7: [41], 8: [43], 11: [40], 12: [23

In [422]:
rank_instance = rankClass()
rank_instance.build_tfidf()
print(rank_instance.tfidf_scores)
#printing the calculated tf-idf scores for all words in the index

defaultdict(<class 'dict'>, {'wikipedia,': {0: 0.003563738717531852, 3: 0.009627044174443685, 7: 0.0009152471574295052}, 'free': {0: 0.002521411961469733, 7: 0.0010792575407259312, 11: 0.0035155170358843234, 14: 0.0035408998303672425, 15: 0.003496717479542696}, 'encyclopediajump': {0: 0.002672804038148889}, 'contentmain': {0: 0.0006041177753801229, 1: 0.00043966663166111845, 4: 0.0018650937668481572, 6: 0.000791251901087097, 7: 0.00010343389728119182, 8: 0.00039462941162530276, 11: 0.0016846008216693034, 12: 0.0010444525094349681, 14: 0.0016967640044972406}, 'menumain': {0: 0.0006041177753801229, 1: 0.00043966663166111845, 4: 0.0018650937668481572, 6: 0.000791251901087097, 7: 0.00010343389728119182, 8: 0.00039462941162530276, 11: 0.0016846008216693034, 12: 0.0010444525094349681, 14: 0.0016967640044972406}, 'menumov': {0: 0.0006041177753801229, 1: 0.00043966663166111845, 4: 0.0018650937668481572, 6: 0.000791251901087097, 7: 0.00010343389728119182, 8: 0.00039462941162530276, 11: 0.001684

# Implementing A Function To Bind Results To Corrosponding URLs

In [424]:
s=searchClass(build_index_instance.wordIndex)
def searchSentenceUrl(sentence):
    visited_links=list(c.visited)
    results=s.searchSentence(sentence)
    for result in results:
        print(visited_links[result[0]])
        print(result[0])

# Examples

In [426]:
searchSentenceUrl('holder')

https://donate.wikimedia.org/?wmf_source=donate&amp;wmf_medium=sidebar&amp;wmf_campaign=foundation.wikimedia.org&amp;uselang=en
14
https://wikimediafoundation.org/about/financial-reports/
11


In [427]:
print(s.orSearch('holder','president'))
print(s.andSearch('hi','log'))
print(s.notSearch('connect','holder'))

[(14, (271,), 0.006043236222280403), (11, (273,), 0.005999915532514952)]
[(0, [26, 436, 478, 481], 0.009279118095181532)]
[(3, [113], 0.006811314257025876), (8, [1612], 0.0008235342174741613), (7, [7009], 0.00021585150814518622)]


In [428]:
print(s.searchSentence('play games on your phone'))
searchSentenceUrl('play games on your phone')

[(7, (1328,), 0.0004576235787147526), (0, (603,), 0.002672804038148889), (0, (280,), 0.002672804038148889)]
https://meta.wikimedia.org/wiki/Special:MyLanguage/Fundraising/Reports
7
https://foundation.wikimedia.org/wiki/Project:General_disclaimer
0
https://foundation.wikimedia.org/wiki/Project:General_disclaimer
0


In [429]:
print(s.searchSentence('trump is the new president'))
searchSentenceUrl('trump is the new president')

[(8, (468, 638), 0.0019532339375410257), (0, (382,), 0.0014950524547630859), (7, (810, 1074, 1881, 2249, 2371, 4174, 4576, 5354, 7455), 0.0023037758116749843), (9, (267,), 0.0020263951390342874)]
https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Universal_Code_of_Conduct
8
https://foundation.wikimedia.org/wiki/Project:General_disclaimer
0
https://meta.wikimedia.org/wiki/Special:MyLanguage/Fundraising/Reports
7
https://stats.wikimedia.org/#/donate.wikimedia.org
9


In [430]:
print(s.searchSentence('trump is the new president'))
searchSentenceUrl('trump is the new president')

[(8, (468, 638), 0.0019532339375410257), (0, (382,), 0.0014950524547630859), (7, (810, 1074, 1881, 2249, 2371, 4174, 4576, 5354, 7455), 0.0023037758116749843), (9, (267,), 0.0020263951390342874)]
https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Universal_Code_of_Conduct
8
https://foundation.wikimedia.org/wiki/Project:General_disclaimer
0
https://meta.wikimedia.org/wiki/Special:MyLanguage/Fundraising/Reports
7
https://stats.wikimedia.org/#/donate.wikimedia.org
9


In [431]:
print(s.searchSentence('trump is the new president'))
searchSentenceUrl('trump is the new president')

[(8, (468, 638), 0.0019532339375410257), (0, (382,), 0.0014950524547630859), (7, (810, 1074, 1881, 2249, 2371, 4174, 4576, 5354, 7455), 0.0023037758116749843), (9, (267,), 0.0020263951390342874)]
https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Universal_Code_of_Conduct
8
https://foundation.wikimedia.org/wiki/Project:General_disclaimer
0
https://meta.wikimedia.org/wiki/Special:MyLanguage/Fundraising/Reports
7
https://stats.wikimedia.org/#/donate.wikimedia.org
9


In [432]:
print(s.searchSentence('new news about the world'))
searchSentenceUrl('new news about the world')

[(8, (300, 401), 0.002811043549238743), (0, (908,), 0.0021516406601178296), (7, (810, 1074, 1881, 2249, 2371, 4174, 4576, 5354, 7455), 0.0023037758116749843), (0, (382,), 0.0014950524547630859), (9, (267,), 0.0020263951390342874), (7, (4318,), 0.0003683927010501038), (7, (4171,), 0.0003683927010501038), (8, (468, 638), 0.0019532339375410257)]
https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Universal_Code_of_Conduct
8
https://foundation.wikimedia.org/wiki/Project:General_disclaimer
0
https://meta.wikimedia.org/wiki/Special:MyLanguage/Fundraising/Reports
7
https://foundation.wikimedia.org/wiki/Project:General_disclaimer
0
https://stats.wikimedia.org/#/donate.wikimedia.org
9
https://meta.wikimedia.org/wiki/Special:MyLanguage/Fundraising/Reports
7
https://meta.wikimedia.org/wiki/Special:MyLanguage/Fundraising/Reports
7
https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Universal_Code_of_Conduct
8


In [433]:
print(s.searchSentence('what is happening in universe'))
searchSentenceUrl('what is happening in universe')

[(7, (4339,), 0.0004576235787147526)]
https://meta.wikimedia.org/wiki/Special:MyLanguage/Fundraising/Reports
7


In [434]:
print(s.searchSentence('i like to eat chocolate'))
#the result is the document id, the positions in that document and the td-idf scores 
searchSentenceUrl('i like to eat chocolate')
#this is to bind the result to the related website

[(1, (964,), 0.0009175203489351976), (9, (338,), 0.0017087617648287913), (4, (110,), 0.003892179575443358), (8, (1183, 1292), 0.0016470684349483226), (7, (849, 934, 1023, 1061, 2063, 2438, 2471, 3164, 3495, 6729, 6772, 7373, 7427), 0.002806069605887421)]
https://stats.wikimedia.org/#/meta.wikimedia.org
1
https://stats.wikimedia.org/#/donate.wikimedia.org
9
https://www.mediawiki.org/wiki/Special:MyLanguage/Developer_Portal/Contribute
4
https://foundation.wikimedia.org/wiki/Special:MyLanguage/Policy:Universal_Code_of_Conduct
8
https://meta.wikimedia.org/wiki/Special:MyLanguage/Fundraising/Reports
7
